# Лекция: Кластерный анализ в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 12** (адаптация с языка R на Python)

## Краткая теория

**Кластерный анализ** — разбиение объектов на группы (кластеры) так, чтобы объекты внутри кластера были похожи, а между кластерами — различались.

Методы:
1. **Иерархические** — дендрограмма, агломеративная кластеризация (`linkage`)
2. **Нейерархические** — k-means, задаём число кластеров заранее

В Python: **scipy.cluster.hierarchy**, **sklearn.cluster.KMeans**, **StandardScaler**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(42)
print("Библиотеки загружены")


---
## 1. Подготовка данных

Перед кластеризацией данные обычно **стандартизируют**.


In [ ]:
tickers = ["GAZP", "LKOH", "VTBR", "SBER", "ROSN", "GMKN",
           "SNGS", "AFLT", "KMAZ", "AVAZ", "OGK1", "OGK2", "OGK5",
           "PMTL", "PLZL", "MTSI", "RASP", "SIBN"]
n = len(tickers)
X = pd.DataFrame({
    "return": np.random.normal(0.05, 0.15, n),
    "volatility": np.abs(np.random.normal(0.2, 0.1, n)),
    "volume": np.random.lognormal(10, 0.5, n),
    "pe": np.random.uniform(5, 25, n),
}, index=tickers)
print(X.round(3))


In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)
print(X_scaled.round(3).head())


---
## 2. Иерархическая кластеризация

В R: `hclust(dist(x), method = "ward.D2")` + `plot(hc)`


In [ ]:
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(Z, labels=X_scaled.index.tolist(), leaf_rotation=90)
plt.title("Дендрограмма (Ward)")
plt.ylabel("Расстояние")
plt.tight_layout()
plt.show()


In [ ]:
k = 4
labels_h = fcluster(Z, t=k, criterion="maxclust")
result_h = pd.DataFrame({"cluster": labels_h}, index=X_scaled.index)
print(result_h.sort_values("cluster"))
print("\nРазмеры кластеров:")
print(result_h["cluster"].value_counts().sort_index())


---
## 3. K-means

В R: `kmeans(x, centers = k)`

Число кластеров: **elbow** (inertia) и **silhouette**.


In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), inertias, "o-")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow method")

axes[1].plot(list(K_range), silhouettes, "o-")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
axes[1].set_title("Silhouette score")
plt.tight_layout()
plt.show()


In [ ]:
k_opt = 4
km = KMeans(n_clusters=k_opt, n_init=25, random_state=42, max_iter=1000)
labels_km = km.fit_predict(X_scaled)

result_km = pd.DataFrame({"cluster": labels_km + 1}, index=X_scaled.index)
print(result_km.sort_values("cluster"))
print("\nРазмеры:", result_km["cluster"].value_counts().sort_index().to_dict())
print("\nЦентры:")
print(pd.DataFrame(km.cluster_centers_, columns=X_scaled.columns).round(3))


In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=labels_km, cmap="tab10", s=80, edgecolors="k")
for i, name in enumerate(X_scaled.index):
    plt.annotate(name, (coords[i, 0], coords[i, 1]), fontsize=8, xytext=(4, 4),
                 textcoords="offset points")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"K-means, k={k_opt}")
plt.colorbar(scatter, label="cluster")
plt.tight_layout()
plt.show()


---
## 4. Задание: KLast1.csv

Иерархическая кластеризация регионов по социально-экономическим показателям.

```python
df = pd.read_csv("KLast1.csv", index_col=0)
X = df.select_dtypes(include=[np.number])
X_scaled = StandardScaler().fit_transform(X)
Z = linkage(X_scaled, method="ward")
dendrogram(Z, labels=df.index.tolist(), leaf_rotation=90)
labels = fcluster(Z, t=5, criterion="maxclust")
df["cluster"] = labels
print(df.groupby("cluster").mean())
```


In [ ]:
np.random.seed(7)
regions = [f"Region_{i}" for i in range(1, 21)]
demo = pd.DataFrame({
    "unemployment": np.random.uniform(3, 15, 20),
    "income": np.random.uniform(20, 80, 20),
    "industry": np.random.uniform(10, 60, 20),
    "science": np.random.uniform(1, 20, 20),
    "investment": np.random.uniform(5, 50, 20),
}, index=regions)
demo.iloc[:7] += [2, -10, -5, -3, -5]
demo.iloc[7:14] += [-2, 15, 10, 5, 10]

X_d = StandardScaler().fit_transform(demo)
Z_d = linkage(X_d, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z_d, labels=demo.index.tolist(), leaf_rotation=90)
plt.title("Демо: дендрограмма регионов")
plt.tight_layout()
plt.show()

labels_d = fcluster(Z_d, t=3, criterion="maxclust")
demo["cluster"] = labels_d
print(demo.groupby("cluster").mean().round(2))


### Как описать результаты

1. По дендрограмме оцените число кластеров.
2. Сравните с elbow / silhouette для k-means.
3. Опишите профили кластеров (средние показателей).

---
## Шпаргалка: R → Python

| Задача в R | Python |
|------------|--------|
| `scale(x)` | `StandardScaler().fit_transform(x)` |
| `hclust(d, method="ward.D2")` | `linkage(X, method="ward")` |
| `plot(hc)` | `dendrogram(Z, labels=...)` |
| `cutree(hc, k=4)` | `fcluster(Z, t=4, criterion="maxclust")` |
| `kmeans(x, 4)` | `KMeans(n_clusters=4).fit(X)` |
| `km$cluster` | `km.labels_` |
| `km$centers` | `km.cluster_centers_` |
| `km$tot.withinss` | `km.inertia_` |

---
## Рекомендации

1. Стандартизируйте признаки перед кластеризацией.
2. Файл **KLast1.csv** подставьте локально.
3. Материалы: https://rpubs.com/AllaT/clust3

**Удачи с выполнением Задания 12!**
